# Zsh History — data exploration

Reads `~/.zsh_history` and counts commands per day.

Only the **first word** of each command is captured to avoid leaking arguments or secrets.

In [ ]:
import sys
sys.path.insert(0, "..")

import polars as pl
from pipeline.extractors.zsh_history import _parse_history, _to_df

In [ ]:
rows = _parse_history()
print(f"{len(rows)} raw records")
rows[:5]

In [ ]:
df = _to_df(rows)
print(df.shape)
df.head(10)

In [ ]:
# Date range
print("From:", df["date"].min())
print("To:  ", df["date"].max())
print("Days:", df["date"].n_unique())

In [ ]:
# Top commands by total count
(
    df.group_by("category")
    .agg(pl.col("value").sum().alias("total"))
    .sort("total", descending=True)
    .head(20)
)

In [ ]:
# Daily total commands
daily = (
    df.group_by("date")
    .agg(pl.col("value").sum().alias("total_commands"))
    .sort("date")
)
print("Avg commands/day:", daily["total_commands"].mean().round(1))
daily.tail(14)